## **Auromatizacion cen Snowflake con Python.**

In [1]:
import snowflake.connector
import os
import configparser

# Conexión a Snowflake
config_path = os.path.join(os.environ['USERPROFILE'], '.snowsql', 'config')
config = configparser.ConfigParser()
config.read(config_path)

# Se obtiene la información desde nuestro archivo de configuración previamente creado.
# El archivo de configuración contiene el nombre de la cuenta, el nimbre de usuario y la contraseña de nuestra cuenta de Snowflake.
try:
    account = config['connections.example']['accountname']
    user = config['connections.example']['username']
    password = config['connections.example']['password']
except KeyError as e:
    print(f'Error: {e}')

# Se realiza la conexión a Snowflake utilizando la información obtenida del archivo de configuración.
try:
    conn = snowflake.connector.connect(
        user = user,
        password = password,
        account = account
    )
    print('Conexión exitosa.')
except Exception as e:
    print(f'Error: {e}')

Conexión exitosa.


In [ ]:
# En esta celda se definen las variables de entorno necesarias para la conexión a Snowflake
# incluyendo el nombre del almacén de datos, la base de datos, el esquema y la tabla que se utilizarán en las consultas.

WAREHOUSE = 'COMPUTE_WH'
DATABASE = 'DEVMOON_SAMPLE'
SCHEMA = 'PUBLIC'
TABLE = 'CAMPAING_DATA_PYTHON'
path_archivo = os.path.join(os.getcwd(), 'dataset.csv').replace('\\', '/')
path_archivo

'c:/Users/yolic/OneDrive/Desktop/SnowflakePracticas/dataset.csv'

In [ ]:
# Se selecciona el almacén de datos, la base de datos y el esquema que se utilizarán en las consultas.

conn.cursor().execute(f'USE WAREHOUSE {WAREHOUSE}')
conn.cursor().execute(f'USE DATABASE {DATABASE}')
conn.cursor().execute(f'USE SCHEMA {SCHEMA}')

In [ ]:
# En esta celda se crea una tabla en Snowflake utilizando la sentencia SQL CREATE TABLE.

sql_ddl = f"""
CREATE OR REPLACE TABLE {DATABASE}.{SCHEMA}.{TABLE} (
    DateStart DATE,
    Campaing STRING,
    Region STRING,
    Clicks NUMBER,
    Impressions NUMBER,
    Views NUMBER,
    Cost NUMBER
);
"""
conn.cursor().execute(sql_ddl)

In [ ]:
# En esta celda se utiliza la sentencia SQL PUT para cargar un archivo CSV desde el sistema de archivos local a Snowflake.

conn.cursor().execute(f'PUT file://{path_archivo} @%{TABLE}')
conn.cursor().execute(f'COPY INTO {TABLE} FILE_FORMAT =(TYPE = CSV SKIP_HEADER = 1)')

In [ ]:
# En esta celda se selecciona el almacén de datos y la base de datos que se utilizarán en las consultas.

cur = conn.cursor()

cur.execute('USE WAREHOUSE COMPUTE_WH;')

cur.execute('USE DATABASE DEVMOON_SAMPLE;')

In [ ]:
# En esta celda se ejecuta una consulta SQL para seleccionar todos los registros de la tabla CAMPAING_DATA_PYTHON y se imprimen los resultados.

sql = """
SELECT *
FROM CAMPAING_DATA_PYTHON
"""
cur.execute(sql)
result = cur.fetchall()
for row in result:
    print(row)

(datetime.date(2023, 12, 24), 'CursosDeProgramacion', 'Oeste', 26148, 133468, 21979, 606)
(datetime.date(2023, 11, 11), 'AprendeCSharpFacil', 'Norte', 34586, 340290, 49146, 495)
(datetime.date(2024, 5, 24), 'CodigoEnEspanol', 'Norte', 20337, 343710, 32831, 1655)
(datetime.date(2023, 9, 5), 'AprendePythonFacil', 'Sur', 45332, 279000, 15840, 991)
(datetime.date(2024, 6, 19), 'CodigoEnEspanol', 'Este', 36817, 462135, 42041, 1784)
(datetime.date(2023, 8, 22), 'AprendeJavaScriptYA', 'Norte', 39359, 440556, 5551, 205)
(datetime.date(2024, 4, 10), 'AprendeJavaScriptYA', 'Norte', 24900, 338969, 8259, 134)
(datetime.date(2023, 10, 6), 'AprendeCSharpFacil', 'Oeste', 8161, 301356, 14102, 742)
(datetime.date(2024, 3, 12), 'CursosDeProgramacion', 'Oeste', 32254, 206296, 6448, 726)
(datetime.date(2024, 1, 23), 'CursoDeJavaAvanzado', 'Este', 17853, 92740, 6913, 458)
(datetime.date(2023, 8, 25), 'MasterEnPython', 'Oeste', 20774, 427615, 19746, 1071)
(datetime.date(2024, 2, 15), 'CursoDeJavaAvanzado', 

In [ ]:
# En esta celda se utiliza la función fetch_pandas_all() para obtener los resultados de la consulta en un DataFrame de pandas.

import pandas as pd
pd = cur.fetch_pandas_all()
pd

,DATESTART,CAMPAING,REGION,CLICKS,IMPRESSIONS,VIEWS,COST
0,2023-12-24,CursosDeProgramacion,Oeste,26148,133468,21979,606
1,2023-11-11,AprendeCSharpFacil,Norte,34586,340290,49146,495
2,2024-05-24,CodigoEnEspanol,Norte,20337,343710,32831,1655
3,2023-09-05,AprendePythonFacil,Sur,45332,279000,15840,991
4,2024-06-19,CodigoEnEspanol,Este,36817,462135,42041,1784
...,...,...,...,...,...,...,...
995,2024-07-06,CursoDeJavaAvanzado,Sur,29588,40026,18668,330
996,2023-12-13,AprendeCSharpFacil,Sur,24523,238325,5276,195
997,2024-01-26,AprendeCSharpFacil,Este,22159,437721,44179,442
998,2024-01-17,AprendeJavaScriptYA,Sur,47854,359512,20689,1674


In [ ]:
# Información sobre el DataFrame

pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   DATESTART    1000 non-null   object
 1   CAMPAING     1000 non-null   object
 2   REGION       1000 non-null   object
 3   CLICKS       1000 non-null   int32 
 4   IMPRESSIONS  1000 non-null   int32 
 5   VIEWS        1000 non-null   int32 
 6   COST         1000 non-null   int16 
dtypes: int16(1), int32(3), object(3)
memory usage: 37.2+ KB
